In [1]:
import sys
sys.path.append("../")
import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import requests
from src.chunking import chunk_text
from src.loaders.txt_loader import load_txt
from src.loaders.pdf_loader import load_pdf
from src.loaders.document_loader import load_documents
from src.pipeline.build_docs import build_docs
from src.pipeline.batch_query import batch_query
from src.vector_store.faiss_store import *
from src.rag.generate import ask_llm
from src.rag.prompt import build_context
from src.retrieval.bm25_store import *
from src.retrieval.hybrid import hybrid_retrieve
from src.config import *
from src.evaluation.logger import save_results
from datetime import datetime
from src.evaluation.chunk_stats import *
from src.evaluation.retrieval_stats import *

d:\Program\anaconda3\envs\rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["HF_HOME"] = HF_CACHE

In [3]:
# 1. embedding模型（轻量）
model = SentenceTransformer(EMBED_MODEL)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7922.78it/s]


In [5]:
loaded_docs = load_documents(DATA_DIR)
docs = build_docs(loaded_docs, MOTHOD, CHUNK_SIZE, OVERLAP)

In [6]:
len(docs)
docs[:20]

[{'chunk_id': 0,
  'source': 'AAA.txt',
  'page': None,
  'text': 'Lorem ipsum style text is commonly used when testing document pipelines and retrieval systems because meaning is less important than structure and length. This paragraph intentionally contains repeated concepts and moderately natural language so that chunk boundaries become easier to inspect during debugging. The purpose is not readability but controlled segmentation. Some sentences are short while others extend slightly to imitate uneven writing patterns found in real documents. Retrieval syste'},
 {'chunk_id': 1,
  'source': 'AAA.txt',
  'page': None,
  'text': 'e others extend slightly to imitate uneven writing patterns found in real documents. Retrieval systems may behave differently depending on overlap values, paragraph preservation, and metadata strategies. Engineers often underestimate how strongly chunk configuration affects later search quality. Therefore this section exists mainly as neutral content with suff

In [7]:
index = build_index(model, docs)
save_index(index, docs, STORAGE_DIR)

In [4]:
index, docs = load_index(STORAGE_DIR)

In [5]:
stats = analyze_chunks(docs)
print_chunk_stats(stats)


=== Chunk Statistics ===
Chunks: 37
Avg length: 383.0
Min length: 101
Max length: 500
Std: 111.2
Tiny (<100): 0

Per source:
AAA.txt: 6
coffee.txt: 4
game_design.txt: 3
sample.pdf: 6
sample.txt: 7
space_exploration.txt: 4
tech_history.txt: 4
urban_transport.txt: 3


In [6]:
bm25 = build_bm25(docs)

In [6]:
for i in range(len(docs)-1):
    print("\n================")
    print("CHUNK", i)
    print(docs[i]["text"][-100:])
    print("NEXT")
    print(docs[i+1]["text"][:100])


CHUNK 0
mainly with energy and productivity, coffee culture varies significantly between regions.

Coffee be
NEXT
e varies significantly between regions.

Coffee begins as a fruit grown in tropical environments. Af

CHUNK 1
ter harvesting, beans go through processing stages that influence sweetness, acidity, and body. Roas
NEXT
 that influence sweetness, acidity, and body. Roasting introduces another layer of variation. Light 

CHUNK 2
roasts tend to preserve more original characteristics, while darker roasts emphasize bitterness and 
NEXT
ics, while darker roasts emphasize bitterness and roasted flavors.

Brewing methods also change the 

CHUNK 3
final result. Pour-over techniques are often valued for clarity and control. Espresso uses pressure 
NEXT
d for clarity and control. Espresso uses pressure to produce concentrated flavor and serves as the b

CHUNK 4
ase for drinks such as cappuccino and latte. Cold brew extracts flavor differently and is commonly p
NEXT
brew extracts flavor 

In [7]:
def retrieve(query, k=2):
    q_emb = model.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    distances, indices = index.search(q_emb, k)
    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "chunk_id": docs[idx]["chunk_id"],
            "source": docs[idx]["source"],
            "score": float(distances[0][rank]),
            "page": docs[idx]["page"],
            "text": docs[idx]["text"]
        })
    return results

In [8]:
def format_score(score):

    if score is None:
        return "-"

    return f"{score:.3f}"

In [9]:
if __name__ == "__main__":
    queries = [
        "What users prefer in AI systems?",
        "What is cosine similarity?",
        "How does RAG work?",
        "Who invented quantum mechanics?"
    ]
    results = batch_query(
        queries,
        lambda q:
        hybrid_retrieve(
            q,
            retrieve,
            bm25,
            docs,
            TOP_K,
            HYBRID_ALPHA
        )
    )
    
    for result in results:
        print("\n===================")
        print(
            f"\nQuestion:\n"
            f"{result['query']}"
        )
        if DEBUG:
            print("\n=== Retrieved ===")
            for r in result["retrieved"]:
                print(
                    f"""
Rank: {r['rank']}
                
Chunk: {r['chunk_id']}

Retriever: {r['retriever']}

Raw faiss score:
{format_score(
r['faiss_raw_score']
)}

Raw bm25 score:
{format_score(
r['bm25_raw_score']
)}

Score: {r['score']:.3f}

Source: {r['source']}

Page: {r['page']}
                
Text: {r['text']}
                    """
                )
        print("\n=== Answer ===")
        print(result["answer"])
        print("\n=== Sources ===")
        seen=set()
        for r in result["retrieved"]:
            source=(
                f"{r['source']}"
                if r["page"] is None
                else
                f"{r['source']} Page {r['page']}"
            )
            if source not in seen:
                print(source)
                seen.add(source)



Question:
What users prefer in AI systems?

=== Answer ===
Users often prefer AI systems that:
• respond quickly,
• explain results clearly,
• maintain predictable behavior,
• allow correction and iteration.
Trust is influenced by transparency and consistency more than raw capability.

=== Sources ===
sample.pdf Page 2
sample.pdf Page 1
urban_transport.txt


Question:
What is cosine similarity?

=== Answer ===
Cosine similarity focuses on vector direction rather than magnitude.

=== Sources ===
sample.txt
AAA.txt


Question:
How does RAG work?

=== Answer ===
RAG works by combining information retrieval with large language model generation. First, documents are collected and preprocessed, then divided into smaller chunks to improve retrieval granularity. Each chunk is converted into a dense vector representation using an embedding model, which is stored in a vector database or search index. When a user submits a query, the same embedding model converts the query into a vector, and th

In [10]:
stats = analyze_retrieval(results)
print_retrieval(stats)


=== Retrieval Statistics ===
Queries: 4
Avg score: 0.518
Min score: 0.006
Max score: 1.0
Unique sources: 6

Top reused chunks:
Chunk 22: 2
Chunk 0: 2
Chunk 18: 1
Chunk 17: 1
Chunk 13: 1


In [11]:
time = datetime.now()
path = (f"../results/"f"{time:%Y%m%d_%H%M%S}"".json")
save_results(results, path)